# Generalization presentation demo

**Question:** Does the published seed-17, epoch-2 LoRA checkpoint lose less answer accuracy than the Qwen3-0.6B base model when approximately 1,024 irrelevant tokens precede the answer passage?

This notebook shows the saved 90- and 810-question results and performs **live inference on one fixed illustrative question**. It does not retrain the model or regenerate the 4,860 validation predictions. The 810-question primary result is a paired EM-change difference of **+3.40 percentage points** (95% article-cluster bootstrap CI **+0.93 to +5.80**). The earlier 90-question interval crossed zero, and its original prediction of an 8-point LoRA decline was not met.

**For a live talk:** Select a Colab GPU runtime. Run cells 1–3 before the presentation and leave the model loaded. During the talk, run cell 4. Cell 5 is an optional counterexample. The first setup may download the ~1.5 GB pinned base model; the 20 MB LoRA adapter is already in the repository. The notebook never mounts Drive.

In [ ]:
# 1. Get the frozen publication and saved generalization results.
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # Set before importing torch.
%pip -q install "transformers==4.53.3" "tokenizers==0.21.4" "peft==0.17.1" "accelerate==1.8.1" "huggingface_hub==0.36.2" "numpy==2.2.6" "safetensors==0.8.0" "pandas==2.2.3" "matplotlib==3.10.0"
from pathlib import Path
import subprocess, sys
ROOT = Path('/content/LLM-generalization')
SOURCE_COMMIT = 'bff11c7de6450bc1b6bd86682764cf621303c0ec'
if not (ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/oscarwyx/LLM-generalization.git', str(ROOT)], check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
sys.path.insert(0, str(ROOT / 'src'))
from genqa.pretrained import verify_published
verified = verify_published(ROOT)  # Checks the adapter and fixed data without a GPU.
assert verified['status'] == 'passed' and verified['counts']['test'] == 900
DATA = ROOT / 'submission/generalization_repro/data'
print('Published adapter and test set verified. Training is not run.')
print('Repository commit:', SOURCE_COMMIT)

In [ ]:
# 2. Display recorded quantitative evidence. This cell needs no GPU.
import pandas as pd
from IPython.display import display, Image, Markdown
initial = pd.read_csv(DATA / 'initial_90/exploratory_interaction.csv')
validation = pd.read_csv(DATA / 'validation_810/paired_effects.csv')
primary = validation[(validation.metric == 'em') & (validation.contrast == 'LoRA_minus_Baseline')].iloc[0]
assert abs(primary.change_pp - 3.395061728395061) < 1e-9
print('Initial 90: exploratory paired difference in EM change (pp)')
display(initial[initial.metric == 'em'].round(2))
print('Remaining 810: paired changes and model interaction (pp)')
display(validation[validation.metric == 'em'].round(2))
display(Markdown('**Interpretation:** the LoRA checkpoint has a smaller EM decline in this SQuAD prepended-context setting. The 810 questions come from the same 45 articles, not a new domain.'))
for label, path in [
    ('Initial 90: length effects', ROOT / 'submission/presentation/overleaf/figures/initial_90_length_effects.png'),
    ('Remaining 810: paired validation', ROOT / 'submission/presentation/overleaf/figures/validation_810_quantitative.png')]:
    display(Markdown('### ' + label))
    display(Image(filename=str(path), width=820))

In [ ]:
# 3. Install exact inference dependencies and load the published base + LoRA checkpoint.
# Colab already supplies torch. Use Runtime > Change runtime type > GPU before this cell.
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime in Colab before loading the model.'
from genqa.pretrained import load_published_model
from genqa import engine
loaded = load_published_model(ROOT)
model, tokenizer = loaded['model'], loaded['tokenizer']
model.eval()
model.requires_grad_(False)
model.config.use_cache = True
print('Ready:', loaded['spec']['model']['model_id'], '| GPU:', torch.cuda.get_device_name(0))
print('Adapter:', loaded['adapter_dir'], '| epoch:', loaded['spec']['training']['epoch'])

## Live inference: one fixed illustrative example

The question below was chosen in advance from the saved validation cases. The live cell runs **four short generations**: original and prepended context, each with the base model and the LoRA adapter. It also shows the recorded predictions for comparison. A single example does not estimate the overall effect; the 810-question table above does that.

In [ ]:
# 4. Run this cell during the talk. No training and no full-test inference.
import ast
from contextlib import nullcontext
from transformers import GenerationConfig

predictions = pd.read_csv(DATA / 'validation_810/all_predictions.csv.gz')
DEMO_ID = '06bc06a0d0734d92bdf0ba8b5ca34e01'  # Fixed example: baseline becomes verbose; LoRA stays exact.

def run_live_case(question_id):
    subset = predictions[(predictions.id == question_id) &
                         (predictions.condition.isin(['original', 'extra_1024_seed_101']))].copy()
    assert len(subset) == 4 and set(subset.method) == {'Baseline', 'LoRA'}
    spec = loaded['spec']['evaluation']
    generation = GenerationConfig(
        do_sample=False, num_beams=1, max_new_tokens=spec['max_new_tokens'],
        use_cache=True, eos_token_id=model.generation_config.eos_token_id,
        pad_token_id=tokenizer.pad_token_id, repetition_penalty=1.0)
    eos_ids = generation.eos_token_id if isinstance(generation.eos_token_id, list) else [generation.eos_token_id]
    output = []
    for method in ('Baseline', 'LoRA'):
        for condition in ('original', 'extra_1024_seed_101'):
            saved = subset[(subset.method == method) & (subset.condition == condition)].iloc[0]
            answers = ast.literal_eval(saved.answers)
            row = {'question': saved.question, 'context': saved.eval_context, 'answers': answers}
            prepared = engine.prepare_input(row, tokenizer, loaded['prompt'], spec['max_input_tokens'])
            ids = torch.tensor([prepared['input_ids']], device='cuda')
            active_adapter = model.disable_adapter() if method == 'Baseline' else nullcontext()
            with active_adapter, torch.inference_mode():
                generated = model.generate(ids, attention_mask=torch.ones_like(ids),
                    generation_config=generation, use_model_defaults=False, do_sample=False)
            tokens = generated[0, ids.shape[1]:].tolist()
            stop = next((i for i, token in enumerate(tokens) if token in eos_ids), len(tokens))
            live = tokenizer.decode(tokens[:stop], skip_special_tokens=True).strip()
            output.append({'method': method, 'condition': condition,
                'prompt_tokens': prepared['input_tokens'], 'live_prediction': live,
                'live_EM': engine.score_answer(live, answers)['em'],
                'recorded_prediction': saved.prediction, 'recorded_EM': 100 * saved.em})
    print('Question:', subset.iloc[0].question)
    print('Accepted answers:', ast.literal_eval(subset.iloc[0].answers))
    display(pd.DataFrame(output))

run_live_case(DEMO_ID)

## Optional second example: a LoRA failure

The next cell shows that LoRA can also lose exact match under added context. Run it only if there is time. Keep the aggregate result and the example separate in your explanation.

In [ ]:
# 5. Optional counterexample. Set to True only if you want a second live comparison.
RUN_SECOND_CASE = False
if RUN_SECOND_CASE:
    run_live_case('2036bd85a0df4d8fa8f770dbc3fe0a09')

## Closing statement

“In our 810-question validation, both models lost accuracy when irrelevant text was prepended. The LoRA checkpoint lost **3.40 fewer EM points** than the base model, with a paired article-bootstrap interval from **+0.93 to +5.80 points**. This supports a narrow robustness claim for this SQuAD setup. It does not show cross-domain transfer, and we observed individual LoRA failures.”